In [0]:
# This will help make our code much simpler. You would not believe how long it took me to find a library that did this
# Another option (much heftier) would be mlextend. This enables us to do time series cross-validation using the
# actual time stamps
# %pip install scikit-lego

# skops is a nicer serialization package for mlflow than the default that comes with this environment.
# it is safer, so we will get in the good habit of using it. It only works with sklearn models and could
# be somewhat restrictive if maybe you do something very custom and fancy. In that case you will have to 
# tell it to trust what you did

 # %pip install skops

In [0]:
# This is just a basic notebook that trains our ML model on the full dataset.

from pyspark.sql import functions as F
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from datetime import timedelta
from sklego.model_selection import TimeGapSplit
from sklearn.model_selection import cross_validate, GridSearchCV

import mlflow.sklearn
from mlflow.models import infer_signature
import io
import mlflow
import pandas as pd
from mlflow import MlflowClient

# We load the predictors
predictors = spark.table("workspace.default.gold_station_predictors")

# We load the targets. The filter is unnecessary at the time of writing
targets = spark.table("workspace.default.gold_station_targets").filter("horizon_minutes = 30")

# We put them together
full_data = predictors.join(targets,["station_id","fetched_at"])

# We list the numeric features

numeric_features_bike = ["num_bikes_available", "bike_available_fraction",
               f"bike_change_30min_per_minute_fraction", "log_capacity", "weekend",
               "hour_sin_1", "hour_cos_1", "hour_sin_2", "hour_cos_2"]

# To make sure this fits in memory, we will take a 10% sample of the training rows. I'll separately show
# a pyspark version of this

SAMPLE_FRACTION = 0.1

data_bike = (full_data.filter(F.col("bike_target").isNotNull()) # Remove null rows
            .select(*numeric_features_bike, "geographic_cell","fetched_at", "bike_target") # Unpack the numerics, select the geographic cells datas, and the target. Discards various labeling information 
            .sample(fraction=SAMPLE_FRACTION, seed=1).toPandas() # Convert from spark to Pandas
        ) 

# This is needed because of a pandas/MLFlow interaction that happened offscreen, basically
# mlflow will infer a signature for our model at the end based on our data. It will
# think we have integers, but pandas casts integer columns with missing data to floats. Apparently
# there is a special Int64 datatype that allows for missing values with floats, but this is beginning
# too make later interactions with MLFlow more complex
data_bike[numeric_features_bike] = data_bike[numeric_features_bike].astype("float64")

# Here is our sklearn pipeline

model_bike = make_pipeline(
    ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_features_bike), # Median impute
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["geographic_cell"]), # One-Hot the geographic cells. Handle unknown lets us use the model as is on data which has a new value of the cells
        ]),
        LogisticRegression(C=0.25, max_iter=200), # Logistic regression, the convergence is slow with so many geo cells
    )

model_bike.fit(data_bike, data_bike["bike_target"])



In [0]:
# To build a similar model for docks, you would just replace "bike" with "dock" in the above

# This is just a basic notebook that trains our ML model on the full dataset.

from pyspark.sql import functions as F
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# We load the predictors
predictors = spark.table("workspace.default.gold_station_predictors")

# We load the targets. The filter is unnecessary at the time of writing
targets = spark.table("workspace.default.gold_station_targets").filter("horizon_minutes = 30")

# We put them together
full_data = predictors.join(targets,["station_id","fetched_at"])

# We list the numeric features

numeric_features_dock = ["num_docks_available", "dock_available_fraction",
               f"dock_change_30min_per_minute_fraction", "log_capacity", "weekend",
               "hour_sin_1", "hour_cos_1", "hour_sin_2", "hour_cos_2"]

# To make sure this fits in memory, we will take a 10% sample of the training rows. I'll separately show
# a pyspark version of this

data_dock = (full_data.filter(F.col("dock_target").isNotNull()) # Remove null rows
            .select(*numeric_features_dock, "geographic_cell", "dock_target") # Unpack the numerics, select the geographic cells datas, and the target. Discards various labeling information 
            .sample(fraction=0.1, seed=1).toPandas() # Convert from spark to Pandas
        ) 

# Here is our sklearn pipeline

model_dock = make_pipeline(
    ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_features_dock), # Median impute
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["geographic_cell"]), # One-Hot the geographic cells. Handle unknown lets us use the model as is on data which has a new value of the cells
        ]),
        LogisticRegression(C=0.25,max_iter=200), # Logistic regression
    )
    

model_dock.fit(data_dock, data_dock["dock_target"])



In [0]:
# Now let's see how things get more complex when we do cross validation


data_bike = data_bike.reset_index(drop=True)   # the splitter matches rows by index

# We are going to start out by trying an expanding window of 7 days

cv = TimeGapSplit(date_serie=data_bike["fetched_at"], # This is the date column
                  train_duration=timedelta(days=30), # This is the minimal size of the training set. THe first fold will be 30 days
                  valid_duration=timedelta(days=7), # This is the size of the validation set. We will train on an expanding window, testing on the next 7 days always (see next line). Rolling/sliding window might also be a good approach, I don't know which is best!
                  gap_duration=timedelta(minutes=40),  # This is to ensure a gap between the border of the training and validation sets. Needed because we are predicting events that happen in the future, and the future might cross the boundary and thus leak
                  window="expanding" # Expandind window means we train on more and more data. This could be bad if there
                  # is data drift (or perhaps just seaonality), or if the data gets so large that the model is too 
                  # expensive to train. In the latter case, with logistic regression we could probably do something called "online training" where we just update the weights when we retrain the model.
                  )


# This will do some hyperparameter tuning
model_bike_cv_search = GridSearchCV(model_bike,
                      param_grid={"logisticregression__C": [0.04, 0.1,0.3, 1.0,3.0,10.0]}, 
                      cv=cv,
                      scoring="neg_log_loss")
model_bike_cv_search.fit(data_bike, data_bike["bike_target"])

In [0]:
# And we can take a basic look at the fit:

import pandas as pd
results = pd.DataFrame(model_bike_cv_search.cv_results_)
split_columns = [c for c in results.columns if c.startswith("split") and c.endswith("_test_score")]


summary = results[["param_logisticregression__C", "rank_test_score", "mean_test_score", "std_test_score", *split_columns]].copy()
score_columns = ["mean_test_score", *split_columns]
summary[score_columns] = -summary[score_columns]   # scikit-learn reports negative log loss
display(summary.sort_values("rank_test_score"))

print("Best C:", model_bike_cv_search.best_params_["logisticregression__C"], "| log loss:", round(-model_bike_cv_search.best_score_, 5))


In [0]:
# Now we are going to use MLFlow. MLFlow is a combination of a few things:
# MLFlow is a server hosted (in our case) by databricks which tracks model training
# runs, storing their outcomes, the values of the model parameters, and keeping copies
# of the actual fitted models. 

# There is also a python library called mlflow which enables you to interact with that server.
# We will now show how to replicate the previous experiment with mlflow tracking/saving turned on

# Where MLflow saves the runs (the experiment) and the models (the registry)
mlflow.set_registry_uri("databricks-uc") # This will save the models in the databricks unity catalog

# This will work in your workspace:
current_user = spark.sql("SELECT current_user()").first()[0] 
# mlflow.set_experiment("/Users/gihchess@gmail.com/citibike-model-training") We could also hardcode the path
# We store the experiment results in the folder of the user, but share the model in the uc
# This is a bit of a moot point for a free edition where the UC isn't really shareable

mlflow.set_experiment(f"/Users/{current_user}/citibike-model-training")

# Unity catalog naming system. The first two parts are catalog and schema. I'm not giving the model
# a stupid name intentionally, but I sort of wish I create a "citibike" schema so names would
# be more sensible and now am locked into "default" until I fix it

model_name = "workspace.default.bike_stockout_lr"
input_columns = numeric_features_bike + ["geographic_cell"] # The columns the model needs to predict

# We begin the model run. This creates an active run object
with mlflow.start_run(run_name="bike_stockout_lr_gridsearch") as run:

    # We fit the model 
    model_bike_cv_search.fit(data_bike, data_bike["bike_target"])

    # We record the settings of this run
    mlflow.log_params({
        "C_grid": str(model_bike_cv_search.param_grid["logisticregression__C"]),
        "train_duration": str(cv.train_duration),
        "valid_duration": str(cv.valid_duration),
        "gap_duration": str(cv.gap_duration),
        "window": cv.window,
        "sample_fraction": SAMPLE_FRACTION,
    })
    mlflow.log_params(model_bike_cv_search.best_params_) # The hyperparameter chosen

    # Record the results. scikit-learn reports negative log loss, so we change the sign
    mlflow.log_metric("cv_log_loss", -model_bike_cv_search.best_score_)
    results = pd.DataFrame(model_bike_cv_search.cv_results_).drop(columns="params")
    mlflow.log_text(results.to_csv(index=False), "cv_results.csv") # The score of every C on every fold

    # Save the best model. GridSearchCV already refit it on all rows with the best C

    best_model = model_bike_cv_search.best_estimator_

    # The input_example gives an example of the data the model uses
    # and also is used to infer the model signature
    input_example = data_bike[input_columns].head(5)
    model_info = mlflow.sklearn.log_model(
        best_model, name="model",
        input_example=input_example,
        signature=infer_signature(input_example, best_model.predict_proba(input_example)),
        pyfunc_predict_fn="predict_proba",   # The saved model returns probabilities, we are principled
        serialization_format="skops",  # Safe choice for storing an sklearn model
        skops_trusted_types=["numpy.dtype"], # Cost of using skops here
        registered_model_name=model_name)    #  Registers a new version of the model

# The scoring notebook loads the version with this alias

# We have given the model an "alias". We will use a model called "champion" to serve predictions
# When we train a new model, we will give it the challenger alias. If we like it more we will promote it
# by hand to "champion", either in a notebook (I'll show the code below)
# or you can actually do using the gui and navigate to the model registry

MlflowClient().set_registered_model_alias(model_name, "challenger", model_info.registered_model_version)
print("Registered version", model_info.registered_model_version)

In [0]:
model_info = mlflow.sklearn.log_model(
        best_model, name="model",
        input_example=input_example,
        signature=infer_signature(input_example, best_model.predict_proba(input_example)),
        pyfunc_predict_fn="predict_proba",   # The saved model returns probabilities, we are principled
        serialization_format="skops",  # Safe choice for storing an sklearn model
        skops_trusted_types=["numpy.dtype"], # Cost of using skops here
        registered_model_name=model_name)    #  Registers a new version of the model

# The scoring notebook loads the version with this alias

# We have given the model an "alias". We will use a model called "champion" to serve predictions
# When we train a new model, we will give it the challenger alias. If we like it more we will promote it
# by hand to "champion", either in a notebook (I'll show the code below)
# or you can actually do using the gui and navigate to the model registry

MlflowClient().set_registered_model_alias(model_name, "challenger", model_info.registered_model_version)
print("Registered version", model_info.registered_model_version)

In [0]:
# We will name our current challenger the champion model. Normally we would do this by comparing the models
# carefully 


client = MlflowClient(registry_uri="databricks-uc")

model_name = "workspace.default.bike_stockout_lr"

challenger = client.get_model_version_by_alias(model_name, "challenger")
client.set_registered_model_alias(model_name, "champion", challenger.version)
client.delete_registered_model_alias(model_name, "challenger")  
print(f"Version {challenger.version} is now the champion")